In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("eda.ipynb"))
PNG_DIR = os.path.join(NOTEBOOK_DIR, "png")
os.makedirs(PNG_DIR, exist_ok=True)

def savefig(name, fig=None):
    f = fig or plt.gcf()
    f.savefig(os.path.join(PNG_DIR, name), dpi=300, bbox_inches="tight")
    plt.close(f)

In [ ]:
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("eda.ipynb"))
BASE_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, ".."))

data = os.path.join(BASE_DIR, "data", "concat_for_eda.csv")

if not os.path.exists(data):
    print(f"[!] {data} not found.")
else:
    print(f"Loading: {data}")

In [ ]:
df= pd.read_csv(data)
print(df)
print(df.shape)

In [ ]:
df.info()

### Descriptive Statistics by Diagnosis Group

In [ ]:
df["anemia_class"].unique().tolist()

In [ ]:
labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

for label in labels:
    subset = df[df["anemia_class"] == label][numeric_cols]
    n = len(subset)
    print(f"{label} (n={n})")
    print(subset.describe().round(2).to_string())

### Missing Value Rate by Diagnosis Group
Heatmap showing the percentage of missing values for each feature within each diagnosis group. Supports the argument for not imputing — each diagnosis group has a different missing data pattern.

In [ ]:
labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

missing_by_diag = pd.DataFrame()
for label in labels:
    subset = df[df["anemia_class"] == label][numeric_cols]
    missing_pct = (subset.isnull().sum() / len(subset) * 100).round(1)
    missing_by_diag[label] = missing_pct

missing_by_diag_filtered = missing_by_diag[missing_by_diag.max(axis=1) > 0]

if missing_by_diag_filtered.empty:
    print("No missing values found — showing full missing-rate table (all zeros).")
    plt.figure(figsize=(10, 8))
    sns.heatmap(missing_by_diag, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5,
                vmin=0, vmax=1)
    plt.title("Missing Value Rate (%) by Diagnosis Group")
    plt.ylabel("Feature")
    plt.tight_layout()
    savefig("missing_by_diagnosis.png")
else:
    plt.figure(figsize=(10, 8))
    sns.heatmap(missing_by_diag_filtered, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5)
    plt.title("Missing Value Rate (%) by Diagnosis Group")
    plt.ylabel("Feature")
    plt.tight_layout()
    savefig("missing_by_diagnosis.png")

print(missing_by_diag)

### Shapiro-Wilk Normality Test
Test normality of each feature within each diagnosis group. If most groups reject normality (p<0.05), this justifies using non-parametric tests (Kruskal-Wallis) instead of ANOVA.

In [ ]:
from scipy.stats import shapiro
import warnings

labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

shapiro_results = []

for col in numeric_cols:
    for label in labels:
        vals = df[df["anemia_class"] == label][col].dropna()
        if len(vals) >= 3:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                stat, p = shapiro(vals)
            shapiro_results.append({
                "Feature": col,
                "Group": label,
                "W-statistic": round(stat, 4),
                "p-value": round(p, 6),
                "Normal": "Yes" if p >= 0.05 else "No"
            })

shapiro_df = pd.DataFrame(shapiro_results)

non_normal_count = shapiro_df[shapiro_df["Normal"] == "No"].shape[0]
total_tests = shapiro_df.shape[0]
print(f"{non_normal_count}/{total_tests} tests reject normality (p<0.05)")
print(f"→ Justifies using Kruskal-Wallis (non-parametric) instead of ANOVA\n")

summary = shapiro_df.pivot_table(index="Feature", columns="Group", values="Normal", aggfunc="first")
print(summary)

### Kruskal-Wallis Test
Compare the distribution of each feature across the diagnosis groups

In [ ]:
from scipy.stats import kruskal
import warnings

labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

kw_results = []

for col in numeric_cols:
    groups = []
    for label in labels:
        vals = df[df["anemia_class"] == label][col].dropna()
        if len(vals) >= 3:
            groups.append(vals)

    if len(groups) >= 2:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            stat, p = kruskal(*groups)

        if not pd.isna(stat):
            kw_results.append({
                "Feature": col,
                "H-statistic": round(stat, 4),
                "p-value": round(p, 6),
                "Significant": "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            })

kw_df = pd.DataFrame(kw_results).sort_values("p-value")
print(kw_df.to_string(index=False))

sig_features = kw_df[kw_df["Significant"] != ""]["Feature"].tolist()
print(f"\nStatistically significant features (p<0.05): {sig_features}")

In [ ]:
import scikit_posthocs as sp
import warnings

labels = ["IDA", "ACD", "ACD/IDA"]
exclude = labels + ["gender_female"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

sig_features = kw_df[kw_df["Significant"] != ""]["Feature"].tolist()

for col in sig_features:
    melted = []
    for label in labels:
        vals = df[df["anemia_class"] == label][col].dropna()
        for v in vals:
            melted.append({"Group": label, "Value": v})

    melted_df = pd.DataFrame(melted)

    if melted_df.empty or melted_df["Group"].nunique() < 2:
        continue

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        dunn = sp.posthoc_dunn(melted_df, val_col="Value", group_col="Group", p_adjust="bonferroni")

    sig_pairs = []
    for i in range(len(dunn.index)):
        for j in range(i + 1, len(dunn.columns)):
            g1, g2 = dunn.index[i], dunn.columns[j]
            p = dunn.iloc[i, j]
            if p < 0.05:
                star = "***" if p < 0.001 else "**" if p < 0.01 else "*"
                sig_pairs.append(f"  {g1} vs {g2}: p={p:.6f} {star}")

    if sig_pairs:
        print(f"\n{col}")
        for pair in sig_pairs:
            print(pair)

### Univariable Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_copy= df.copy()

X_y_don= df_copy.drop(columns= ["ACD", "IDA"])


In [ ]:
null_X = X_y_don.drop(columns=["anemia_class"])

null_counts = null_X.notnull().sum()

plt.figure(figsize=(10, 5))
null_counts.plot(kind='bar')

plt.xticks(rotation=80)

plt.title('Number of Non-null values per Column')
plt.xlabel('Columns')
plt.ylabel('Count of Non-null Values')

plt.tight_layout()
savefig("nonnull_counts.png")

In [ ]:
null_X = X_y_don.drop(columns=["anemia_class"])

null_counts = null_X.notnull().sum()

gender_counts = X_y_don["gender_female"].value_counts()
gender_counts.index = ["Female" if v else "Male" for v in gender_counts.index]

null_counts = null_counts.drop("gender_female")
null_counts = pd.concat([null_counts, gender_counts])

plt.figure(figsize=(10, 5))
null_counts.plot(kind="bar")

plt.xticks(rotation=80)

plt.title("Number of Non-null values per Column (with Gender breakdown)")
plt.xlabel("Columns")
plt.ylabel("Count")

plt.tight_layout()
savefig("nonnull_with_gender.png")

In [ ]:
high_missing = X_y_don.drop(columns=["anemia_class"]).columns[X_y_don.drop(columns=["anemia_class"]).isnull().mean() > 0.5].tolist()
drop_cols = [c for c in high_missing + ["gender_female"] if c in X_y_don.columns]
X_y_don_notnull = X_y_don.drop(columns=drop_cols)

for col_name in X_y_don_notnull.columns:
    if col_name != "anemia_class":
        fig, ax = plt.subplots()
        sns.boxplot(
            data=X_y_don_notnull,
            x="anemia_class",
            y=col_name,
            hue="anemia_class",
            legend=False,
            ax=ax
        )
        ax.set_title(f"Boxplot: {col_name} by Diagnosis")
        savefig(f"boxplot_{col_name}.png", fig)

In [ ]:
fig, ax = plt.subplots()
gender_labels = X_y_don["gender_female"].map({True: "Female", False: "Male"})
sns.countplot(x=gender_labels, ax=ax)
ax.set_xlabel("Gender")
ax.set_ylabel("Count")
ax.set_title("Gender Distribution")
savefig("gender_distribution.png", fig)

In [ ]:
plot_df = X_y_don.copy()
plot_df["gender"] = plot_df["gender_female"].map({True: "Female", False: "Male"})

fig, ax = plt.subplots()
sns.countplot(data=plot_df, x="gender", hue="anemia_class", ax=ax)
ax.legend(
    title="Distribution of gender by diagnosis",
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
ax.set_title("Gender by Diagnosis Group")
savefig("gender_by_diagnosis.png", fig)

### Correlation Matrix — CBC Parameters Only

In [ ]:
cbc_cols = ["RBC", "hemoglobin", "MCV", "MCHC", "rdw_cv"]
cbc_available = [c for c in cbc_cols if c in df.columns]

corr_cbc = df[cbc_available].corr()

n_matrix_cbc = pd.DataFrame(index=cbc_available, columns=cbc_available, dtype=int)
for c1 in cbc_available:
    for c2 in cbc_available:
        n_matrix_cbc.loc[c1, c2] = df[[c1, c2]].dropna().shape[0]

annot_cbc = corr_cbc.copy().astype(str)
for c1 in cbc_available:
    for c2 in cbc_available:
        r = corr_cbc.loc[c1, c2]
        n = n_matrix_cbc.loc[c1, c2]
        annot_cbc.loc[c1, c2] = f"{r:.2f}\n(n={n})"

mask = np.triu(np.ones_like(corr_cbc, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_cbc, mask=mask, annot=annot_cbc, fmt="",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
    cbar_kws={"shrink": 0.8}, annot_kws={"size": 9}
)
ax.set_title("Correlation Matrix — CBC Parameters (with pairwise n)")
savefig("corr_cbc_only.png", fig)

print("Pairwise sample sizes:")
print(n_matrix_cbc.to_string())

### Correlation Matrix — CBC + Iron/Inflammation Parameters

In [ ]:
iron_cols = ["Ferritin", "CRP", "Fe", "TIBC", "tsat"]
all_cols = cbc_cols + iron_cols
all_available = [c for c in all_cols if c in df.columns]

corr_full = df[all_available].corr()

n_matrix_full = pd.DataFrame(index=all_available, columns=all_available, dtype=int)
for c1 in all_available:
    for c2 in all_available:
        n_matrix_full.loc[c1, c2] = df[[c1, c2]].dropna().shape[0]

annot_full = corr_full.copy().astype(str)
for c1 in all_available:
    for c2 in all_available:
        r = corr_full.loc[c1, c2]
        n = n_matrix_full.loc[c1, c2]
        if pd.isna(r):
            annot_full.loc[c1, c2] = f"NaN\n(n={n})"
        else:
            annot_full.loc[c1, c2] = f"{r:.2f}\n(n={n})"

mask_full = np.triu(np.ones_like(corr_full, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(14, 11))
sns.heatmap(
    corr_full, mask=mask_full, annot=annot_full, fmt="",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    linewidths=0.5, square=True, ax=ax,
    cbar_kws={"shrink": 0.8}, annot_kws={"size": 8}
)
ax.set_title("Correlation Matrix — CBC + Iron/Inflammation (with pairwise n)")
savefig("corr_cbc_iron.png", fig)

print("Pairwise sample sizes:")
print(n_matrix_full.to_string())

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["anemia_class"], ax=ax)
ax.set_title("Diagnosis Class Distribution")
plt.xticks(rotation=0)
savefig("diagnosis_distribution.png", fig)

In [ ]:
y = df[["ACD", "IDA"]]

counts = y.value_counts()

print(counts)

In [ ]:
labels_binary = ["ACD", "IDA"]

co_matrix = pd.DataFrame(0, index=labels_binary, columns=labels_binary)

for _, row in y.iterrows():
    active_labels = row[row == 1].index.tolist()
    for l1 in active_labels:
        for l2 in active_labels:
            co_matrix.loc[l1, l2] += 1

print("Co-occurrence matrix:")
print(co_matrix)

### Model Comparison — Friedman Test
Train all models (XGBoost, LightGBM, CatBoost), collect CV fold scores, and run Friedman test + Nemenyi post-hoc to compare.

In [ ]:
import sys, os

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

train_path = os.path.join(BASE_DIR, "data", "train_set_reduced_features.csv")
test_path = os.path.join(BASE_DIR, "data", "test_set_reduced_features.csv")

if not os.path.exists(train_path) or not os.path.exists(test_path):
    print(f"[skip] Training/test data not found. Run train/test split first.")
else:
    from src.dataclass.modelfactory import ModelFactory, models_config
    from src.functions.function import friedman_compare, print_result

    df_train = pd.read_csv(train_path)
    df_test = pd.read_csv(test_path)

    X_train = df_train.drop(columns=["ACD", "IDA"])
    y_train = df_train[["ACD", "IDA"]]
    X_test = df_test.drop(columns=["ACD", "IDA"])
    y_test = df_test[["ACD", "IDA"]]

    all_cv_scores = {}

    for name in models_config:
        print(f"\n{'='*40}")
        print(f"Training: {name}")
        print(f"{'='*40}")
        factory = ModelFactory(name)
        result = factory.train_and_evaluate(X_train, y_train, X_test, y_test)
        print_result(result)
        all_cv_scores[name] = result["cv_fold_scores"]

    print(f"\n{'='*60}")
    print("Friedman Test — Comparing all models")
    print(f"{'='*60}")
    friedman_compare(all_cv_scores)